In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from collections import Counter

pd.set_option("display.max_columns", None)

SEED = 42

In [5]:
train = pd.read_csv("../Data/raw/train.csv")
test = pd.read_csv("../Data/raw/test.csv")
attendance = pd.read_csv("../Data/raw/Attendance_series.csv")

In [6]:
print(train.shape)
print(test.shape)
print(attendance.shape)

(12000, 18)
(3000, 17)
(1048575, 5)


In [7]:
train_original = train.copy()
test_original = test.copy()
attendance_original = attendance.copy()

In [8]:
print(train["dropout_risk"].value_counts())

print(
    train["dropout_risk"]
    .value_counts(normalize=True)
)

dropout_risk
0    7200
1    3000
2    1800
Name: count, dtype: int64
dropout_risk
0    0.60
1    0.25
2    0.15
Name: proportion, dtype: float64


In [9]:
train = train_original.copy()
test = test_original.copy()
attendance = attendance_original.copy()

In [10]:
def evaluate_oof(y_true, probs):

    preds = np.argmax(
        probs,
        axis=1
    )

    score = f1_score(
        y_true,
        preds,
        average="macro"
    )

    print("Macro F1:", score)

    return score

In [11]:
experiment_log = []

def log_result(name, score):

    experiment_log.append(
        {
            "experiment": name,
            "score": score
        }
    )

    return pd.DataFrame(
        experiment_log
    ).sort_values(
        "score",
        ascending=False
    )

In [12]:
BEST_BASELINE = 0.70716

log_result(
    "Current Ensemble",
    BEST_BASELINE
)

,experiment,score
0,Current Ensemble,0.70716


In [13]:
train = train_original.copy()
test = test_original.copy()
attendance = attendance_original.copy()

In [14]:
te_cols = [
    "branch",
    "family_income",
    "parent_education",
    "hostel_status",
    "gender"
]

In [15]:
from sklearn.model_selection import StratifiedKFold

def multiclass_target_encoding(
    train_df,
    test_df,
    col,
    target
):

    train_df = train_df.copy()
    test_df = test_df.copy()

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for cls in [0, 1, 2]:

        oof = np.zeros(len(train_df))

        global_mean = (
            train_df[target] == cls
        ).mean()

        for train_idx, valid_idx in skf.split(
            train_df,
            train_df[target]
        ):

            fold_train = train_df.iloc[train_idx]

            mapping = (
                (fold_train[target] == cls)
                .groupby(fold_train[col])
                .mean()
            )

            oof[valid_idx] = (
                train_df.iloc[valid_idx][col]
                .map(mapping)
                .fillna(global_mean)
            )

        train_df[f"{col}_te_{cls}"] = oof

        full_mapping = (
            (train_df[target] == cls)
            .groupby(train_df[col])
            .mean()
        )

        test_df[f"{col}_te_{cls}"] = (
            test_df[col]
            .map(full_mapping)
            .fillna(global_mean)
        )

    return train_df, test_df

In [16]:
for col in te_cols:

    print("Encoding:", col)

    train, test = multiclass_target_encoding(
        train,
        test,
        col,
        "dropout_risk"
    )

Encoding: branch
Encoding: family_income
Encoding: parent_education
Encoding: hostel_status
Encoding: gender


In [17]:
[c for c in train.columns if "_te_" in c][:20]

['branch_te_0',
 'branch_te_1',
 'branch_te_2',
 'family_income_te_0',
 'family_income_te_1',
 'family_income_te_2',
 'parent_education_te_0',
 'parent_education_te_1',
 'parent_education_te_2',
 'hostel_status_te_0',
 'hostel_status_te_1',
 'hostel_status_te_2',
 'gender_te_0',
 'gender_te_1',
 'gender_te_2']

In [18]:
print(len([c for c in train.columns if "_te_" in c]))
print([c for c in train.columns if "_te_" in c][:10])

15
['branch_te_0', 'branch_te_1', 'branch_te_2', 'family_income_te_0', 'family_income_te_1', 'family_income_te_2', 'parent_education_te_0', 'parent_education_te_1', 'parent_education_te_2', 'hostel_status_te_0']


In [19]:
attendance_stats = (
    attendance
    .groupby("student_id")["attendance_pct"]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

attendance_stats.columns = [
    "student_id",
    "attendance_mean",
    "attendance_std",
    "attendance_min",
    "attendance_max"
]

attendance_stats["attendance_range"] = (
    attendance_stats["attendance_max"]
    - attendance_stats["attendance_min"]
)

train = train.merge(
    attendance_stats,
    on="student_id",
    how="left"
)

test = test.merge(
    attendance_stats,
    on="student_id",
    how="left"
)

In [20]:
cgpa_cols = [
    "cgpa_sem1",
    "cgpa_sem2",
    "cgpa_sem3",
    "cgpa_sem4"
]

for df in [train, test]:

    df["cgpa_mean"] = df[cgpa_cols].mean(axis=1)

    df["cgpa_std"] = df[cgpa_cols].std(axis=1)

    df["cgpa_trend"] = (
        df["cgpa_sem4"]
        - df["cgpa_sem1"]
    )

In [21]:
backlog_cols = [
    "backlogs_sem1",
    "backlogs_sem2",
    "backlogs_sem3"
]

for df in [train, test]:

    df["backlog_total"] = (
        df[backlog_cols]
        .sum(axis=1)
    )

    df["backlog_mean"] = (
        df[backlog_cols]
        .mean(axis=1)
    )

    df["backlog_trend"] = (
        df["backlogs_sem3"]
        - df["backlogs_sem1"]
    )

In [22]:
target = "dropout_risk"

features = [
    c
    for c in train.columns
    if c not in [
        "student_id",
        target,
        "counsellor_note"
    ]
]

In [23]:
cat_features = [
    "branch",
    "gender",
    "hostel_status",
    "family_income",
    "parent_education"
]

In [24]:
def run_cat_cv(
    train_df,
    features,
    cat_features
):

    X = train_df[features]
    y = train_df["dropout_risk"]

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    oof = np.zeros(
        (len(train_df), 3)
    )

    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y)
    ):

        print(f"Fold {fold+1}")

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(
            iterations=1000,
            depth=6,
            learning_rate=0.03,
            loss_function="MultiClass",
            eval_metric="TotalF1",
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        oof[valid_idx] = (
            model.predict_proba(
                X_valid
            )
        )

    return evaluate_oof(
        y,
        oof
    )

In [26]:
for col in cat_features:

    train[col] = (
        train[col]
        .fillna("Missing")
        .astype(str)
    )

In [27]:
for col in cat_features:
    print(
        col,
        train[col].isna().sum()
    )

branch 0
gender 0
hostel_status 0
family_income 0
parent_education 0


In [28]:
score_te = run_cat_cv(
    train,
    features,
    cat_features
)

log_result(
    "Target Encoding + CatBoost",
    score_te
)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Macro F1: 0.5167092131011563


,experiment,score
0,Current Ensemble,0.707160
1,Target Encoding + CatBoost,0.516709


In [29]:
log_result(
    "Target Encoding Failed",
    0.5167
)

,experiment,score
0,Current Ensemble,0.707160
1,Target Encoding + CatBoost,0.516709
2,Target Encoding Failed,0.516700


In [31]:
print(attendance.columns)

Index(['student_id', 'semester', 'week', 'subject', 'attendance_pct'], dtype='object')


In [33]:
subject_features = (
    attendance
    .groupby(["student_id", "subject"])["attendance_pct"]
    .mean()
    .reset_index()
)

In [34]:
subject_features = (
    subject_features
    .groupby("student_id")["attendance_pct"]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

subject_features.columns = [
    "student_id",
    "subject_att_mean",
    "subject_att_std",
    "subject_att_min",
    "subject_att_max"
]

subject_features["subject_att_range"] = (
    subject_features["subject_att_max"]
    -
    subject_features["subject_att_min"]
)

In [35]:
train_exp = train.copy()
test_exp = test.copy()

train_exp = train_exp.merge(
    subject_features,
    on="student_id",
    how="left"
)

test_exp = test_exp.merge(
    subject_features,
    on="student_id",
    how="left"
)

In [36]:
print(
    [c for c in train_exp.columns
     if "subject_att" in c]
)

['subject_att_mean', 'subject_att_std', 'subject_att_min', 'subject_att_max', 'subject_att_range']


In [37]:
week_features = (
    attendance
    .groupby(["student_id", "week"])["attendance_pct"]
    .mean()
    .reset_index()
)

In [38]:
week_features = (
    week_features
    .groupby("student_id")["attendance_pct"]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

week_features.columns = [
    "student_id",
    "week_att_mean",
    "week_att_std",
    "week_att_min",
    "week_att_max"
]

week_features["week_att_range"] = (
    week_features["week_att_max"]
    -
    week_features["week_att_min"]
)

In [39]:
train_exp = train_exp.merge(
    week_features,
    on="student_id",
    how="left"
)

test_exp = test_exp.merge(
    week_features,
    on="student_id",
    how="left"
)

In [40]:
new_cols = [
    c
    for c in train_exp.columns
    if "subject_att" in c
    or "week_att" in c
]

print(len(new_cols))
print(new_cols)

10
['subject_att_mean', 'subject_att_std', 'subject_att_min', 'subject_att_max', 'subject_att_range', 'week_att_mean', 'week_att_std', 'week_att_min', 'week_att_max', 'week_att_range']


In [41]:
risk_check = train_exp.groupby(
    "dropout_risk"
)[
    [
        "subject_att_std",
        "week_att_std"
    ]
].mean()

print(risk_check)

              subject_att_std  week_att_std
dropout_risk                               
0                    0.034617      0.062446
1                    0.035312      0.062773
2                    0.035139      0.062466


In [43]:
print(train.columns.tolist())

['student_id', 'branch', 'gender', 'hostel_status', 'family_income', 'parent_education', 'scholarship', 'part_time_job', 'commute_time_mins', 'screen_time_hours', 'cgpa_sem1', 'cgpa_sem2', 'cgpa_sem3', 'cgpa_sem4', 'backlogs_sem1', 'backlogs_sem2', 'backlogs_sem3', 'dropout_risk', 'branch_te_0', 'branch_te_1', 'branch_te_2', 'family_income_te_0', 'family_income_te_1', 'family_income_te_2', 'parent_education_te_0', 'parent_education_te_1', 'parent_education_te_2', 'hostel_status_te_0', 'hostel_status_te_1', 'hostel_status_te_2', 'gender_te_0', 'gender_te_1', 'gender_te_2', 'attendance_mean', 'attendance_std', 'attendance_min', 'attendance_max', 'attendance_range', 'cgpa_mean', 'cgpa_std', 'cgpa_trend', 'backlog_total', 'backlog_mean', 'backlog_trend']


In [44]:
[c for c in train.columns if "note" in c.lower()]

[]

In [46]:
import os

for root, dirs, files in os.walk("../data"):
    for file in files:
        print(file)

Attendance_series.csv
Counsellor_notes.csv
Data_dictionary.csv
sample_submission.csv
solution (1).csv
test.csv
train.csv


In [47]:
print(train_original.columns.tolist())

['student_id', 'branch', 'gender', 'hostel_status', 'family_income', 'parent_education', 'scholarship', 'part_time_job', 'commute_time_mins', 'screen_time_hours', 'cgpa_sem1', 'cgpa_sem2', 'cgpa_sem3', 'cgpa_sem4', 'backlogs_sem1', 'backlogs_sem2', 'backlogs_sem3', 'dropout_risk']


In [51]:
notes = pd.read_csv("../Data/raw/Counsellor_notes.csv")

print(notes.shape)
print(notes.columns.tolist())
notes.head()

(15000, 2)
['student_id', 'counsellor_note']


,student_id,counsellor_note
0,STU00001,Student is performing well. Follow-up required.
1,STU00002,Needs to improve focus in class. Action plan d...
2,STU00003,"Struggling slightly with core subjects, advise..."
3,STU00004,Student expressed some stress regarding course...
4,STU00005,Multiple backlogs. Demotivated. Action plan di...


In [52]:
for col in notes.columns:
    print(col)

student_id
counsellor_note


In [53]:
train_nlp = train_original.copy()
test_nlp = test_original.copy()

In [54]:
train_nlp = train_nlp.merge(
    notes,
    on="student_id",
    how="left"
)

test_nlp = test_nlp.merge(
    notes,
    on="student_id",
    how="left"
)

In [55]:
print(train_nlp.shape)
print(test_nlp.shape)

print(
    train_nlp["counsellor_note"]
    .isna()
    .sum()
)

(12000, 19)
(3000, 18)
0


In [56]:
train_nlp = train_original.copy()
test_nlp = test_original.copy()

train_nlp = train_nlp.merge(
    notes,
    on="student_id",
    how="left"
)

test_nlp = test_nlp.merge(
    notes,
    on="student_id",
    how="left"
)

In [57]:
train_nlp["counsellor_note"] = (
    train_nlp["counsellor_note"]
    .fillna("")
    .astype(str)
)

test_nlp["counsellor_note"] = (
    test_nlp["counsellor_note"]
    .fillna("")
    .astype(str)
)

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

train_tfidf = tfidf.fit_transform(
    train_nlp["counsellor_note"]
)

test_tfidf = tfidf.transform(
    test_nlp["counsellor_note"]
)

print(train_tfidf.shape)

(12000, 177)


In [59]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=64,
    random_state=42
)

train_svd = svd.fit_transform(
    train_tfidf
)

test_svd = svd.transform(
    test_tfidf
)

print(train_svd.shape)

(12000, 64)


In [60]:
svd_cols = [
    f"note_svd_{i}"
    for i in range(64)
]

train_svd_df = pd.DataFrame(
    train_svd,
    columns=svd_cols
)

test_svd_df = pd.DataFrame(
    test_svd,
    columns=svd_cols
)

In [61]:
X = train_svd_df.copy()
y = train_original["dropout_risk"]

In [62]:
oof_nlp = np.zeros((len(X),3))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold,(train_idx,valid_idx) in enumerate(
    skf.split(X,y)
):

    print(f"Fold {fold+1}")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1500,
        depth=6,
        learning_rate=0.03,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train
    )

    oof_nlp[valid_idx] = (
        model.predict_proba(X_valid)
    )

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [63]:
print(notes["counsellor_note"].nunique())

54


In [64]:
notes["counsellor_note"].value_counts().head(20)

counsellor_note
Good academic-social balance. Action plan discussed.                        471
Good academic-social balance. Follow-up required.                           459
Participates actively in class. Situation monitored.                        456
Participates actively in class. Follow-up required.                         452
Participates actively in class. Action plan discussed.                      451
No further action needed. Action plan discussed.                            451
No further action needed. Situation monitored.                              450
No major issues reported. Situation monitored.                              447
No major issues reported. Action plan discussed.                            447
Discussed minor career goals, doing fine. Action plan discussed.            443
Good academic-social balance. Situation monitored.                          442
Student is performing well. Situation monitored.                            441
No further action needed

In [65]:
print(
    tfidf.get_feature_names_out()[:100]
)

['absence' 'absence noted' 'academic' 'academic social' 'academics'
 'academics action' 'academics follow' 'academics situation' 'action'
 'action needed' 'action plan' 'actively' 'actively class' 'advised'
 'advised tutoring' 'affecting' 'affecting academics' 'attendance'
 'attendance month' 'attendance unresponsive' 'backlogs'
 'backlogs demotivated' 'balance' 'balance action' 'balance follow'
 'balance situation' 'briefly' 'briefly action' 'briefly follow'
 'briefly situation' 'career' 'career goals' 'challenges'
 'challenges affecting' 'class' 'class action' 'class follow'
 'class situation' 'concerns' 'concerns mentioned' 'contact'
 'contact notified' 'core' 'core subjects' 'coursework'
 'coursework action' 'coursework follow' 'coursework situation'
 'demotivated' 'demotivated action' 'demotivated follow'
 'demotivated situation' 'discussed' 'discussed dropping'
 'discussed minor' 'doing' 'doing fine' 'dropping' 'dropping financial'
 'emails' 'emails action' 'emails follow' 'email

In [66]:
print(notes["counsellor_note"].nunique())

54


In [67]:
note_stats = (
    train_nlp
    .groupby("counsellor_note")["dropout_risk"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

print(note_stats.head())

dropout_risk                                               0         1  \
counsellor_note                                                          
Discussed dropping out due to financial stress....  0.000000  0.336449   
Discussed dropping out due to financial stress....  0.000000  0.267241   
Discussed dropping out due to financial stress....  0.000000  0.351064   
Discussed minor career goals, doing fine. Actio...  0.916898  0.083102   
Discussed minor career goals, doing fine. Follo...  0.920354  0.079646   

dropout_risk                                               2  
counsellor_note                                               
Discussed dropping out due to financial stress....  0.663551  
Discussed dropping out due to financial stress....  0.732759  
Discussed dropping out due to financial stress....  0.648936  
Discussed minor career goals, doing fine. Actio...  0.000000  
Discussed minor career goals, doing fine. Follo...  0.000000  


In [68]:
note_risk = (
    train_nlp
    .groupby("counsellor_note")["dropout_risk"]
    .mean()
    .sort_values()
)

print(note_risk.head(20))
print(note_risk.tail(20))

counsellor_note
Discussed minor career goals, doing fine. Follow-up required.       0.079646
Good academic-social balance. Action plan discussed.                0.080000
No major issues reported. Follow-up required.                       0.080556
No further action needed. Situation monitored.                      0.080780
Good academic-social balance. Follow-up required.                   0.081081
Participates actively in class. Situation monitored.                0.082192
Participates actively in class. Follow-up required.                 0.082192
Discussed minor career goals, doing fine. Action plan discussed.    0.083102
Student is performing well. Action plan discussed.                  0.086053
Student is performing well. Follow-up required.                     0.088643
Good academic-social balance. Situation monitored.                  0.090361
No major issues reported. Action plan discussed.                    0.096866
Student is performing well. Situation monitored.            

In [69]:
from sklearn.model_selection import StratifiedKFold

train_note = train_original.copy()
test_note = test_original.copy()

train_note = train_note.merge(
    notes,
    on="student_id",
    how="left"
)

test_note = test_note.merge(
    notes,
    on="student_id",
    how="left"
)

In [70]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_note["note_risk_oof"] = 0.0

In [71]:
for train_idx, valid_idx in skf.split(
    train_note,
    train_note["dropout_risk"]
):

    fold_train = train_note.iloc[train_idx]

    mapping = (
        fold_train
        .groupby("counsellor_note")["dropout_risk"]
        .mean()
    )

    train_note.loc[
        train_note.index[valid_idx],
        "note_risk_oof"
    ] = (
        train_note.iloc[valid_idx]
        ["counsellor_note"]
        .map(mapping)
        .fillna(
            train_note["dropout_risk"].mean()
        )
    )

In [72]:
full_mapping = (
    train_note
    .groupby("counsellor_note")["dropout_risk"]
    .mean()
)

test_note["note_risk_oof"] = (
    test_note["counsellor_note"]
    .map(full_mapping)
    .fillna(
        train_note["dropout_risk"].mean()
    )
)

In [73]:
print(
    train_note.groupby(
        "dropout_risk"
    )["note_risk_oof"].mean()
)

dropout_risk
0    0.225937
1    0.813862
2    1.406686
Name: note_risk_oof, dtype: float64


In [74]:
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

X = train_note[["note_risk_oof"]]
y = train_note["dropout_risk"]

oof = np.zeros((len(X),3))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for train_idx, valid_idx in skf.split(X,y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    model = CatBoostClassifier(
        iterations=500,
        depth=4,
        loss_function="MultiClass",
        verbose=False
    )

    model.fit(
        X_train,
        y_train
    )

    oof[valid_idx] = model.predict_proba(
        X_valid
    )

In [77]:
preds = np.argmax(
    oof,
    axis=1
)

score = f1_score(
    y,
    preds,
    average="macro"
)

print(score)

0.6767904764418261


In [78]:
print(score)

print(
    train_note.groupby(
        "dropout_risk"
    )["note_risk_oof"].mean()
)

0.6767904764418261
dropout_risk
0    0.225937
1    0.813862
2    1.406686
Name: note_risk_oof, dtype: float64


In [79]:
train_final = train.copy()
test_final = test.copy()

train_final["note_risk_oof"] = train_note["note_risk_oof"]

test_final["note_risk_oof"] = test_note["note_risk_oof"]

In [80]:
features_final = [
    c
    for c in train_final.columns
    if c not in [
        "student_id",
        "dropout_risk"
    ]
]

In [81]:
print(len(features_final))
print(features_final[-10:])

43
['attendance_min', 'attendance_max', 'attendance_range', 'cgpa_mean', 'cgpa_std', 'cgpa_trend', 'backlog_total', 'backlog_mean', 'backlog_trend', 'note_risk_oof']


In [82]:
score_note_cat = run_cat_cv(
    train_final,
    features_final,
    cat_features
)

print(score_note_cat)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Macro F1: 0.706984663812433
0.706984663812433
